# GPU Experiment 6: Representation Saturation & Activation Drift Diagnostics

This notebook empirically investigates the representation saturation hypothesis by tracking activation L2 norms $\|h_8^{(t)}\|_2$ and cosine similarity drift $\cos(h_8^{(t)}, h_8^{(0)})$ across generation steps $t \in [1, 200]$ for:
1. **Unsteered Baseline**
2. **Continuous Steering ($K=\infty, \alpha_0=18.0$)**
3. **Hard Cutoff Steering ($K=16, \alpha_0=18.0$)**
4. **Linear Decay Steering ($K=16, \alpha_0=18.0$)**


In [ ]:
!pip install -q evaluate bert_score bitsandbytes accelerate transformers

import os, sys, json, time, math, torch
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device 0: {torch.cuda.get_device_name(0)}")


In [ ]:
possible_paths = [
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

with open(data_path, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

test_subset = full_dataset[-100:]  # N=100 subset for high-resolution diagnostic tracking
train_pool = full_dataset[:-2205]

model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
model.eval()
print("✅ Model loaded for diagnostic tracking!")


In [ ]:
pos_acts, neg_acts = [], []
for item in train_pool[:150]:
    q = item['question']
    pos_ans = item.get('right_answer', item.get('positive_answer'))
    neg_ans = item['hallucinated_answer']
    
    t_pos = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}"
    t_neg = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}"
    
    with torch.no_grad():
        inp_pos = tokenizer(t_pos, return_tensors="pt").to(model.device)
        out_pos = model(inp_pos.input_ids, output_hidden_states=True)
        pos_acts.append(out_pos.hidden_states[8][0, -1, :].detach().cpu())
        
        inp_neg = tokenizer(t_neg, return_tensors="pt").to(model.device)
        out_neg = model(inp_neg.input_ids, output_hidden_states=True)
        neg_acts.append(out_neg.hidden_states[8][0, -1, :].detach().cpu())

v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
v_steer = v_diff / v_diff.norm(p=2)
print("✅ Vector v_steer extracted!")


In [ ]:
target_layer = model.model.layers[8]

def run_diagnostic_tracking(schedule_type="decay", alpha_0=18.0, K=16):
    l2_norms_per_step = [[] for _ in range(100)]  # 100 max new tokens
    
    for item in tqdm(test_subset, desc=f"Tracking {schedule_type}"):
        prompt = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        step_counter = 0
        step_norms = []
        
        def track_hook(module, input_tensor, output_tensor):
            nonlocal step_counter
            step_counter += 1
            cur_t = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
            
            if schedule_type == "continuous":
                alpha_t = alpha_0
            elif schedule_type == "cutoff":
                alpha_t = alpha_0 if step_counter <= K else 0.0
            elif schedule_type == "decay":
                alpha_t = alpha_0 * (1.0 - (step_counter - 1) / K) if 1 <= step_counter <= K else 0.0
            else:
                alpha_t = 0.0
                
            if alpha_t != 0.0:
                v_curr = v_steer.to(device=cur_t.device, dtype=cur_t.dtype)
                cur_t = cur_t + alpha_t * v_curr
                
            norm_val = float(cur_t[0, -1, :].norm(p=2).detach().cpu())
            step_norms.append(norm_val)
            
            if isinstance(output_tensor, tuple):
                return (cur_t,) + output_tensor[1:]
            return cur_t
            
        hook = target_layer.register_forward_hook(track_hook)
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        hook.remove()
        
        for s_idx, nv in enumerate(step_norms[:100]):
            l2_norms_per_step[s_idx].append(nv)
            
    mean_norms = [float(np.mean(l2_norms_per_step[t])) if len(l2_norms_per_step[t]) > 0 else 0.0 for t in range(100)]
    return mean_norms

schedules = ["baseline", "continuous", "cutoff", "decay"]
diagnostic_results = {}

for s in schedules:
    diagnostic_results[s] = run_diagnostic_tracking(schedule_type=s)
    print(f"Completed tracking {s}: Mean Norm at t=10 is {diagnostic_results[s][10]:.2f}, at t=50 is {diagnostic_results[s][50]:.2f}")


In [ ]:
with open("representation_saturation_diagnostics.json", "w", encoding="utf-8") as f:
    json.dump(diagnostic_results, f, indent=2, ensure_ascii=False)

print("✅ Saved representation_saturation_diagnostics.json successfully!")
